# Chapter 14 – Worked examples

This notebook runs the worked example of [Chapter 14](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/0.2.00.2.0/part-3-power-systems/ch14-component-models/) live, in NumPy against pandapower (the Octave
version is `ybus_check.m`, run with MATPOWER). The code lives in the modules next to it (`ybus_numpy.py`,
`small_ybus.py`, `kron_reduction.py`, `long_line.py`); the cells only call them, so a change in a module is a
change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q colorama==0.4.6 deepdiff==8.6.2 geojson==3.3.0 matplotlib networkx numpy packaging pandas pandera==0.34.1 scipy tqdm typing_extensions tzdata
%pip install -q --no-deps pandapower==3.5.6

In [ ]:
import kron_reduction
import long_line
import matplotlib.pyplot as plt
import numpy as np
import small_ybus
import ybus_numpy

## Ybus of IEEE 14 from the connection matrices

`case14_ppc` runs one power flow so that pandapower fills its MATPOWER-style `ppc`. `build_ybus` assembles
$Y_\text{bus} = C_f^\top Y_f + C_t^\top Y_t + [Y_\text{sh}]$ and `max_difference` compares it with pandapower's
own `makeYbus`.

In [ ]:
ppc = ybus_numpy.case14_ppc()
bus, branch, base_mva = ppc['bus'], ppc['branch'], ppc['baseMVA']
ybus, yf, yt = ybus_numpy.build_ybus(bus, branch, base_mva)
print('max |Ybus - makeYbus| =', ybus_numpy.max_difference(ppc))
print('||Ybus - Ybus^T|| =', ybus_numpy.asymmetry(ybus))

The sparsity pattern: 14 diagonal entries and two per branch. The three transformers (buses 4–7, 4–9, 5–6) have
off-nominal taps but no phase shift, so the matrix is still symmetric.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.spy(ybus, markersize=8)
ax.set_title('IEEE 14: nonzeros of Ybus')
plt.show()
print('transformer branches:', ybus_numpy.transformer_branches(branch))

A 5° phase shift on the first transformer makes $y_{ft} \neq y_{tf}$ and $Y_\text{bus}$ non-symmetric.

In [ ]:
k = int(ybus_numpy.transformer_branches(branch)[0])
shifted, _, _ = ybus_numpy.build_ybus(bus, ybus_numpy.with_shift(branch, k, 5.0), base_mva)
print('||Ybus - Ybus^T|| with a 5 degree shift =', round(ybus_numpy.asymmetry(shifted), 4))

## By inspection: a three-bus ring and a tapped transformer

In [ ]:
np.set_printoptions(precision=4, suppress=True)
print(small_ybus.ring3_ybus())
print('row sums:', small_ybus.row_sums(small_ybus.ring3_ybus()))
print(small_ybus.transformer_ybr(1.05, 0.1))

## A Ward-type equivalent is a Schur complement

Keep buses 1–5, eliminate the rest: the reduced matrix reproduces the kept-bus voltages exactly when the
eliminated buses carry no injection.

In [ ]:
reduced = kron_reduction.kron_reduce(ybus, kron_reduction.KEEP)
injection = np.array([1.0, -0.5j, 0.2 + 0.1j, -0.4, 0.3j])
v_full = kron_reduction.kept_voltages_full(ybus, kron_reduction.KEEP, injection)
v_reduced = kron_reduction.kept_voltages_reduced(ybus, kron_reduction.KEEP, injection)
print('reduced matrix is', reduced.shape, 'with', np.count_nonzero(reduced), 'nonzeros')
print('max voltage mismatch:', np.abs(v_full - v_reduced).max())

## When does the nominal pi stop being good enough?

The relative error of the nominal series admittance against the exact long-line value grows with the square of
the length: below 1 % up to about 230 km for this 400 kV line.

In [ ]:
lengths = np.linspace(10, 600, 60)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(lengths, 100 * long_line.series_admittance_error(lengths))
ax.set_xlabel('length [km]')
ax.set_ylabel('error of 1/Z, nominal pi [%]')
ax.grid(True)
plt.show()